# **Healthcare Readmission Consulting Project:**
## **Data Profiling & Data Quality Assessment**

**Purpose:** To assess source data quality, identify material issues that could affect downstream analysis and reporting, and establish appropriate methods for documenting and addressing these issues during ETL.

### Table of Contents

### 1. Project Setup
- 1.1 Import Libraries
- 1.2 Load Source Datasets
- 1.3 Create Dataset Dictionary
- 1.4 Verify Successful Data Loading

### 2. Data Inventory
- 2.1 Dataset-Level Profile
- 2.2 Column-Level Profile
- 2.3 Summary Statistics

### 3. Data Quality Assessment
- 3.1 Numeric Value Validation
- 3.2 Categorical and Ordinal Value Validation
- 3.3 Date Validation
- 3.4 Identifier Validation
- 3.4.1 Leading/Trailing Space Check
- 3.5 Hospital Uniqueness Check

### 4. Data Quality Summary
- Consolidated Validation Results


### 1. Project Setup


In [29]:
# 1.1 Libraries

import re
import numpy as np
import pandas as pd

from IPython.display import display

In [30]:
# 1.2 Load Source Datasets
DATA_PATH = "/content"

df_patients = pd.read_csv(f"{DATA_PATH}/raw_patients.csv")
df_admissions = pd.read_csv(f"{DATA_PATH}/raw_admissions.csv")
df_diagnoses = pd.read_csv(f"{DATA_PATH}/raw_diagnoses.csv")
df_billing = pd.read_csv(f"{DATA_PATH}/raw_billing.csv")
df_hospitals = pd.read_csv(f"{DATA_PATH}/raw_hospitals.csv")

In [31]:
# 1.3 Create Dataset Dictionary
datasets = {
    "Patients": df_patients,
    "Admissions": df_admissions,
    "Diagnoses": df_diagnoses,
    "Billing": df_billing,
    "Hospitals": df_hospitals,
}

In [32]:
# 1.4 Verify Successful Data Loading

for dataset_name, df in datasets.items():
    print("=" * 80)
    print(f"{dataset_name} Dataset")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("=" * 80)

    display(df.head())

    print()

Patients Dataset
Shape: 86,400 rows × 8 columns


,patient_id,age,gender,state,bpl_card,insurance_type,comorbidity_count,prev_admissions
0,2b179f89-9af7-45cf-86b4-3e01a47a3b7b,36,F,Telangana,False,Private,0,0
1,ecaeb9f3-d263-4712-b9b6-3312af5626ca,56,F,Uttar Pradesh,False,ESI,0,0
2,0160a5b6-877e-41e6-b84a-a2a51b367a36,67,F,Telangana,False,ESI,3,0
3,216875fa-b49d-4c47-8c8d-7191ba34ac06,53,M,Haryana,False,Ayushman,3,0
4,a2f3181d-e3f2-4fd3-8057-05a492ffcfb2,60,M,Maharashtra,True,Ayushman,4,1



Admissions Dataset
Shape: 120,000 rows × 17 columns


,admission_id,patient_id,admit_date,discharge_date,los_days,admit_type,ward_type,hospital_id,discharge_type,num_procedures,charlson_index,hba1c,creatinine,haemoglobin,systolic_bp,readmitted_30d,readmitted_7d
0,bcb311d6-0808-498f-8ae1-11abe1fbc08e,6960d399-8398-4db4-9a90-0b329b81bc2b,2024-01-23,2024-01-27,4,Elective,General,b22e7b51-2ea6-4611-a8e5-d3d24ec779d8,Recovered,0,0,6.3,0.86,16.2,157,0,0
1,996ac6db-a192-46a0-b536-503a1c57994b,ec1999d4-f097-4881-91a5-6f9b1f8dac78,2019-11-18,2019-11-19,1,OPD,General,8b1e01b7-9f0e-413f-b330-454d271a5c6e,Recovered,1,0,5.2,1.23,13.1,156,0,0
2,cc4849fa-b740-487b-beda-3086a18273c0,ed56abac-a0ba-44db-a816-b5da4183b9b5,2019-11-22,2019-11-27,5,Emergency,General,eae7c9ec-65c8-4d48-92f3-7da7b3e2b227,Recovered,0,2,5.6,0.67,10.9,130,1,0
3,b38cdd03-165c-428c-b221-9f1d4cf8bdeb,90c335d5-dd71-46c8-acd2-656653f421c6,2020-03-05,2020-03-16,11,Emergency,ICU,df4431ea-1acf-4e77-963c-6173e9787151,Recovered,1,2,4.7,1.40,8.8,159,1,0
4,6a154ec0-92cc-43f5-ada5-ede0a600ce9e,694988e8-9ef5-43e8-aefe-c6b98b1e96fe,2018-05-30,2018-06-04,5,Emergency,General,8b1e01b7-9f0e-413f-b330-454d271a5c6e,Recovered,1,5,5.0,0.94,11.3,173,1,0



Diagnoses Dataset
Shape: 271,341 rows × 6 columns


,diag_id,admission_id,icd10_code,diag_desc,diag_rank,diag_category
0,3b448b98-6b78-4c7c-94c1-f832581c5052,bcb311d6-0808-498f-8ae1-11abe1fbc08e,E87,Metabolic / electrolyte disorder,1,Endocrine
1,137c0404-f1b6-4aca-95f5-2217812a6975,996ac6db-a192-46a0-b536-503a1c57994b,J45,Asthma,1,Respiratory
2,4bfb76a1-12ed-4cbd-b905-34d4bf3196f0,cc4849fa-b740-487b-beda-3086a18273c0,G40,Epilepsy,1,Neurological
3,754ec3d2-e940-4342-9e1d-d88a9acec626,cc4849fa-b740-487b-beda-3086a18273c0,A15,Pulmonary tuberculosis,2,Infectious
4,f4fa0d7b-0a68-4dd9-b7cf-db15c6bf3812,cc4849fa-b740-487b-beda-3086a18273c0,I48,Atrial fibrillation,3,Cardiovascular



Billing Dataset
Shape: 120,000 rows × 6 columns


,bill_id,admission_id,total_cost_inr,govt_subsidy_inr,out_of_pocket_inr,cost_category
0,68e19242-4126-4a7a-afa2-9e6035cdcffa,bcb311d6-0808-498f-8ae1-11abe1fbc08e,18492,16630,1862,Lab
1,b740202e-1d7a-45b4-9575-21b35e219e37,996ac6db-a192-46a0-b536-503a1c57994b,1001,0,1001,Lab
2,f60fecaa-e769-42ea-bee8-58198bdd1208,cc4849fa-b740-487b-beda-3086a18273c0,59696,48650,11046,Room
3,32f41176-c088-4e9a-aa81-ae40f91b912a,b38cdd03-165c-428c-b221-9f1d4cf8bdeb,72588,0,72588,Lab
4,cf80fba9-7fec-4b2e-b548-037a43dcf93c,6a154ec0-92cc-43f5-ada5-ede0a600ce9e,9510,0,9510,Lab



Hospitals Dataset
Shape: 33 rows × 6 columns


,hospital_id,name,state,tier,beds,teaching
0,df0f948a-2202-459a-ba6f-7ffd7dedcc18,Government General Hospital Maharashtra,Maharashtra,tier2,314,True
1,3d59f4a6-ce9c-4945-8f1f-5929f09a7836,Government General Hospital Uttar Pradesh,Uttar Pradesh,tier2,481,True
2,aac12881-4dea-42f3-bf32-8add9d5eee49,Government General Hospital Tamil Nadu,Tamil Nadu,tier2,428,True
3,28879338-cd09-48bb-bedb-730b8ee8ab4e,Government General Hospital Karnataka,Karnataka,tier2,304,True
4,2f5c9dbb-cef7-46a2-8f35-ac5bdc0232c6,Government General Hospital West Bengal,West Bengal,tier2,632,True


### 2. Data Inventory



In [33]:
# 2.1 Dataset-Level Profile

def profile_dataset(df, dataset_name):
    total_cells = df.shape[0] * df.shape[1]
    missing_count = df.isna().sum().sum()

    return {
        "Dataset": dataset_name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": missing_count,
        "Missing (%)": round(
            missing_count / total_cells * 100, 2
        ) if total_cells else 0,
        "Duplicate Rows": df.duplicated().sum(),
        "Numeric Columns": len(
            df.select_dtypes(include=np.number).columns
        ),
        "Categorical Columns": len(
            df.select_dtypes(include="object").columns
        ),
        "Date Columns": len(
            df.select_dtypes(include="datetime").columns
        ),
    }


profile_summary = pd.DataFrame(
    [
        profile_dataset(df, name)
        for name, df in datasets.items()
    ]
)

display(profile_summary)

,Dataset,Rows,Columns,Missing Values,Missing (%),Duplicate Rows,Numeric Columns,Categorical Columns,Date Columns
0,Patients,86400,8,24715,3.58,0,3,4,0
1,Admissions,120000,17,0,0.00,0,9,8,0
2,Diagnoses,271341,6,0,0.00,0,1,5,0
3,Billing,120000,6,0,0.00,0,3,3,0
4,Hospitals,33,6,0,0.00,0,1,4,0


In [34]:
print("Transposed Profile Summary")

profile_summary_transposed = (
    profile_summary
    .set_index("Dataset")
    .T
)

display(profile_summary_transposed)

Transposed Profile Summary


Dataset,Patients,Admissions,Diagnoses,Billing,Hospitals
Rows,86400.00,120000.0,271341.0,120000.0,33.0
Columns,8.00,17.0,6.0,6.0,6.0
Missing Values,24715.00,0.0,0.0,0.0,0.0
Missing (%),3.58,0.0,0.0,0.0,0.0
Duplicate Rows,0.00,0.0,0.0,0.0,0.0
Numeric Columns,3.00,9.0,1.0,3.0,1.0
Categorical Columns,4.00,8.0,5.0,3.0,4.0
Date Columns,0.00,0.0,0.0,0.0,0.0


In [35]:
# 2.2 Column-Level Profile

info_tables = []

for dataset_name, df in datasets.items():

    temp = pd.DataFrame({
        "Dataset": dataset_name,
        "Column": df.columns,
        "Non-Null": df.notna().sum().values,
        "Null": df.isna().sum().values,
        "Dtype": df.dtypes.astype(str).values,
    })

    temp["Null (%)"] = (
        temp["Null"] / len(df) * 100
    ).round(2)

    info_tables.append(temp)


info_summary = pd.concat(
    info_tables,
    ignore_index=True
)

display(info_summary)

,Dataset,Column,Non-Null,Null,Dtype,Null (%)
0,Patients,patient_id,86400,0,object,0.00
1,Patients,age,86400,0,int64,0.00
2,Patients,gender,86400,0,object,0.00
3,Patients,state,86400,0,object,0.00
4,Patients,bpl_card,86400,0,bool,0.00
5,Patients,insurance_type,61685,24715,object,28.61
6,Patients,comorbidity_count,86400,0,int64,0.00
7,Patients,prev_admissions,86400,0,int64,0.00
8,Admissions,admission_id,120000,0,object,0.00
9,Admissions,patient_id,120000,0,object,0.00


In [36]:
# 2.3 Summary Statistic

for dataset_name, df in datasets.items():

    print("=" * 80)
    print(f"Summary Statistics: {dataset_name}")
    print("=" * 80)

    display(
        df.describe(include="all").T
    )

    print()

Summary Statistics: Patients


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
patient_id,86400,86400,b2868cc6-69ef-4b73-8b5f-2cee38a3e617,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,86400.0,NaN,NaN,NaN,48.000405,21.690553,0.0,36.0,51.0,63.0,95.0
gender,86400,3,M,44190,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,86400,15,Maharashtra,9633,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bpl_card,86400,2,False,54790,NaN,NaN,NaN,NaN,NaN,NaN,NaN
insurance_type,61685,3,Ayushman,25967,NaN,NaN,NaN,NaN,NaN,NaN,NaN
comorbidity_count,86400.0,NaN,NaN,NaN,1.42647,1.506211,0.0,0.0,1.0,2.0,8.0
prev_admissions,86400.0,NaN,NaN,NaN,0.896076,1.083745,0.0,0.0,1.0,1.0,9.0



Summary Statistics: Admissions


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
admission_id,120000,120000,a9d5a2bc-3312-449c-851c-4d284fbad4b1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
patient_id,120000,64873,dd0ca7f5-a2e4-4077-8602-49690edaca7e,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
admit_date,120000,3652,2019-07-06,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
discharge_date,120000,3677,2020-07-01,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
los_days,120000.0,NaN,NaN,NaN,6.852708,5.708269,1.0,3.0,5.0,9.0,90.0
admit_type,120000,3,Emergency,66540,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ward_type,120000,4,General,76768,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hospital_id,120000,33,a86650ea-3517-4285-bc11-ffa2085a8df2,3736,NaN,NaN,NaN,NaN,NaN,NaN,NaN
discharge_type,120000,4,Recovered,87420,NaN,NaN,NaN,NaN,NaN,NaN,NaN
num_procedures,120000.0,NaN,NaN,NaN,1.523742,1.542197,0.0,0.0,1.0,2.0,15.0



Summary Statistics: Diagnoses


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
diag_id,271341,271341,d57296a5-6ff6-453c-bc2c-66a777a319d7,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
admission_id,271341,120000,bcca7977-7c90-4bc4-89c7-3d8711e82b96,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
icd10_code,271341,34,E10,12079,NaN,NaN,NaN,NaN,NaN,NaN,NaN
diag_desc,271341,34,Type 1 diabetes mellitus,12079,NaN,NaN,NaN,NaN,NaN,NaN,NaN
diag_rank,271341.0,NaN,NaN,NaN,1.926867,0.998618,1.0,1.0,2.0,3.0,4.0
diag_category,271341,11,Cardiovascular,53410,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Summary Statistics: Billing


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
bill_id,120000,120000,068bca68-136f-4dcd-900f-ee211a160769,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
admission_id,120000,120000,a9d5a2bc-3312-449c-851c-4d284fbad4b1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_cost_inr,120000.0,NaN,NaN,NaN,95779.53165,195511.816741,649.0,12023.0,30970.5,89407.0,6674008.0
govt_subsidy_inr,120000.0,NaN,NaN,NaN,47606.167783,106237.10924,0.0,0.0,12145.5,45855.75,4606120.0
out_of_pocket_inr,120000.0,NaN,NaN,NaN,48173.363867,139551.927859,0.0,0.0,7636.0,33341.0,4648583.0
cost_category,120000,4,Pharmacy,36280,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Summary Statistics: Hospitals


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
hospital_id,33,33,df0f948a-2202-459a-ba6f-7ffd7dedcc18,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
name,33,26,Apollo/Manipal (Private) Andhra Pradesh,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,33,15,Andhra Pradesh,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tier,33,3,tier2,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
beds,33.0,NaN,NaN,NaN,440.454545,294.238964,61.0,191.0,423.0,632.0,1018.0
teaching,33,2,True,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 3. Data Quality Assessment

## Validation Framework

The following rules are applied based on the fields and business rules defined for this project.

### Numeric validation

- **Age:** 0–120
- **BMI:** 10–80
- **Total bill/cost:** ≥ 0
- **Length of stay:** ≥ 0

### Categorical and ordinal validation

Review distinct values for:

- Unexpected categories
- Spelling inconsistencies
- Capitalization inconsistencies
- Ordinal values outside the expected range

### Date validation

- Dates must be parseable.
- Discharge date must not precede admission date.

### Identifier validation

The project specification describes identifiers using the following structure:

`8-4-4-4-12`

> **Technical note:** The identifier check validates the specified string pattern. It does not claim that the identifiers are cryptographically valid UUID v4 values.

3.1 Numeric Value Validation

The checks below report observed minimum and maximum values. These outputs are compared with the defined validation rules to identify potential out-of-range observations.

The profiling step does not automatically classify every clinical measurement as invalid because acceptable ranges have not been explicitly defined for all variables.

In [37]:
# --------------------------------------------------
# Patients
# --------------------------------------------------

print("Patients — Age")

print("Minimum age:", df_patients["age"].min())
print("Maximum age:", df_patients["age"].max())

non_integer_age = df_patients[
    df_patients["age"].notna()
    &
    (df_patients["age"] % 1 != 0)
]

print(
    "Non-integer age values:",
    len(non_integer_age)
)

print()


# --------------------------------------------------
# Admissions
# --------------------------------------------------

print("Admissions — Numeric Fields")

for column in [
    "los_days",
    "hba1c",
    "creatinine",
    "haemoglobin",
    "systolic_bp"
]:

    print(f"{column}:")
    print(f"  Min: {df_admissions[column].min()}")
    print(f"  Max: {df_admissions[column].max()}")

print()


# --------------------------------------------------
# Billing
# --------------------------------------------------

print("Billing — Financial Fields")

for column in [
    "total_cost_inr",
    "govt_subsidy_inr",
    "out_of_pocket_inr"
]:

    print(f"{column}:")
    print(f"  Min: {df_billing[column].min()}")
    print(f"  Max: {df_billing[column].max()}")

print()


# --------------------------------------------------
# Hospitals
# --------------------------------------------------

print("Hospitals — Beds")

print(
    "Minimum beds:",
    df_hospitals["beds"].min()
)

print(
    "Maximum beds:",
    df_hospitals["beds"].max()
)

Patients — Age
Minimum age: 0
Maximum age: 95
Non-integer age values: 0

Admissions — Numeric Fields
los_days:
  Min: 1
  Max: 90
hba1c:
  Min: 4.0
  Max: 14.3
creatinine:
  Min: 0.4
  Max: 15.0
haemoglobin:
  Min: 4.0
  Max: 18.0
systolic_bp:
  Min: 70
  Max: 220

Billing — Financial Fields
total_cost_inr:
  Min: 649
  Max: 6674008
govt_subsidy_inr:
  Min: 0
  Max: 4606120
out_of_pocket_inr:
  Min: 0
  Max: 4648583

Hospitals — Beds
Minimum beds: 61
Maximum beds: 1018


In [38]:
# 3.2 Categorical and Ordinal Value Validation

def check_distinct_values(df, columns):

    for column in columns:

        print("=" * 80)
        print(f"Column: {column}")
        print("=" * 80)

        distinct_values = (
            df[column]
            .dropna()
            .unique()
        )

        print(
            f"Number of distinct values: "
            f"{len(distinct_values)}"
        )

        print("Distinct values:")

        print(
            sorted(
                distinct_values,
                key=str
            )
        )

        print()

In [39]:
print("=" * 80)
print("PATIENTS")
print("=" * 80)

check_distinct_values(
    df_patients,
    [
        "gender",
        "state",
        "bpl_card",
        "insurance_type",
        "comorbidity_count",
        "prev_admissions",
    ],
)

PATIENTS
Column: gender
Number of distinct values: 3
Distinct values:
['F', 'M', 'Other']

Column: state
Number of distinct values: 15
Distinct values:
['Andhra Pradesh', 'Bihar', 'Gujarat', 'Haryana', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Odisha', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Telangana', 'Uttar Pradesh', 'West Bengal']

Column: bpl_card
Number of distinct values: 2
Distinct values:
[np.False_, np.True_]

Column: insurance_type
Number of distinct values: 3
Distinct values:
['Ayushman', 'ESI', 'Private']

Column: comorbidity_count
Number of distinct values: 9
Distinct values:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

Column: prev_admissions
Number of distinct values: 10
Distinct values:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]



In [40]:
print("=" * 80)
print("ADMISSIONS")
print("=" * 80)

check_distinct_values(
    df_admissions,
    [
        "admit_type",
        "ward_type",
        "discharge_type",
        "num_procedures",
        "charlson_index",
        "readmitted_30d",
        "readmitted_7d",
    ],
)

ADMISSIONS
Column: admit_type
Number of distinct values: 3
Distinct values:
['Elective', 'Emergency', 'OPD']

Column: ward_type
Number of distinct values: 4
Distinct values:
['General', 'HDU', 'ICU', 'NICU']

Column: discharge_type
Number of distinct values: 4
Distinct values:
['Expired', 'LAMA', 'Recovered', 'Referred']

Column: num_procedures
Number of distinct values: 15
Distinct values:
[np.int64(0), np.int64(1), np.int64(10), np.int64(11), np.int64(12), np.int64(14), np.int64(15), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

Column: charlson_index
Number of distinct values: 7
Distinct values:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

Column: readmitted_30d
Number of distinct values: 2
Distinct values:
[np.int64(0), np.int64(1)]

Column: readmitted_7d
Number of distinct values: 2
Distinct values:
[np.int64(0), np.int64(1)]



In [41]:
print("=" * 80)
print("DIAGNOSES")
print("=" * 80)

check_distinct_values(
    df_diagnoses,
    [
        "icd10_code",
        "diag_rank",
        "diag_category",
        "diag_desc",
    ],
)

DIAGNOSES
Column: icd10_code
Number of distinct values: 34
Distinct values:
['A09', 'A15', 'A41', 'A91', 'B54', 'C18', 'C34', 'C50', 'E10', 'E11', 'E87', 'G35', 'G40', 'I10', 'I21', 'I48', 'I50', 'I63', 'J18', 'J44', 'J45', 'J96', 'K57', 'K70', 'K80', 'N18', 'N39', 'O34', 'O80', 'P07', 'P22', 'S06', 'S72', 'T14']

Column: diag_rank
Number of distinct values: 4
Distinct values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Column: diag_category
Number of distinct values: 11
Distinct values:
['Cardiovascular', 'Endocrine', 'Gastrointestinal', 'Genitourinary', 'Infectious', 'Injury', 'Neoplasm', 'Neurological', 'Obstetric', 'Perinatal', 'Respiratory']

Column: diag_desc
Number of distinct values: 34
Distinct values:
['Acute myocardial infarction', 'Alcoholic liver disease', 'Asthma', 'Atrial fibrillation', 'COPD', 'Cerebral infarction', 'Cholelithiasis', 'Chronic kidney disease', 'Dengue haemorrhagic fever', 'Disorders — short gestation', 'Diverticular disease of colon', 'Epilepsy

In [42]:
print("=" * 80)
print("BILLING")
print("=" * 80)

check_distinct_values(
    df_billing,
    [
        "cost_category"
    ],
)

BILLING
Column: cost_category
Number of distinct values: 4
Distinct values:
['Lab', 'Pharmacy', 'Procedure', 'Room']



In [43]:
print("=" * 80)
print("HOSPITALS")
print("=" * 80)

check_distinct_values(
    df_hospitals,
    [
        "name",
        "state",
        "tier",
        "teaching",
    ],
)

HOSPITALS
Column: name
Number of distinct values: 26
Distinct values:
['Apollo/Manipal (Private) Andhra Pradesh', 'Apollo/Manipal (Private) Karnataka', 'Apollo/Manipal (Private) Telangana', 'Apollo/Manipal (Private) Uttar Pradesh', 'District Hospital Bihar', 'District Hospital Gujarat', 'District Hospital Kerala', 'District Hospital Madhya Pradesh', 'District Hospital Odisha', 'District Hospital Rajasthan', 'District Hospital Uttar Pradesh', 'Government General Hospital Andhra Pradesh', 'Government General Hospital Bihar', 'Government General Hospital Gujarat', 'Government General Hospital Haryana', 'Government General Hospital Karnataka', 'Government General Hospital Kerala', 'Government General Hospital Madhya Pradesh', 'Government General Hospital Maharashtra', 'Government General Hospital Odisha', 'Government General Hospital Punjab', 'Government General Hospital Rajasthan', 'Government General Hospital Tamil Nadu', 'Government General Hospital Telangana', 'Government General Hospi

## 3.3 Date Validation

Admission and discharge dates are temporarily converted using `errors="coerce"` so that unparseable values can be identified without changing the original source columns.

Three checks are performed:

1. Invalid admission dates
2. Invalid discharge dates
3. Discharge dates earlier than admission dates

In [44]:
# Convert dates temporarily for validation

admit_date = pd.to_datetime(
    df_admissions["admit_date"],
    errors="coerce"
)

discharge_date = pd.to_datetime(
    df_admissions["discharge_date"],
    errors="coerce"
)

In [45]:
# --------------------------------------------------
# Identify invalid admission dates
# --------------------------------------------------

invalid_admit_date = df_admissions[
    admit_date.isna()
    &
    df_admissions["admit_date"].notna()
]


# --------------------------------------------------
# Identify invalid discharge dates
# --------------------------------------------------

invalid_discharge_date = df_admissions[
    discharge_date.isna()
    &
    df_admissions["discharge_date"].notna()
]


# --------------------------------------------------
# Check logical date order
# --------------------------------------------------

invalid_date_order = df_admissions[
    admit_date.notna()
    &
    discharge_date.notna()
    &
    (discharge_date < admit_date)
]

In [46]:
date_validation_summary = pd.DataFrame({

    "Check": [
        "Invalid admit_date",
        "Invalid discharge_date",
        "Discharge before admit",
    ],

    "Count": [
        len(invalid_admit_date),
        len(invalid_discharge_date),
        len(invalid_date_order),
    ],
})

display(date_validation_summary)

,Check,Count
0,Invalid admit_date,0
1,Invalid discharge_date,0
2,Discharge before admit,0


In [47]:
# Display records only when issues are identified

if len(invalid_admit_date) > 0:

    print("Invalid admission dates:")

    display(
        invalid_admit_date[
            [
                "admission_id",
                "admit_date"
            ]
        ]
    )


if len(invalid_discharge_date) > 0:

    print("Invalid discharge dates:")

    display(
        invalid_discharge_date[
            [
                "admission_id",
                "discharge_date"
            ]
        ]
    )


if len(invalid_date_order) > 0:

    print("Discharge date earlier than admission date:")

    display(
        invalid_date_order[
            [
                "admission_id",
                "admit_date",
                "discharge_date"
            ]
        ]
    )

## 3.4 Identifier Validation

The project specification uses an `8-4-4-4-12` identifier format.

The function below checks the expected structure across primary-key and foreign-key fields.

### Identifier fields

**Patients**
- `patient_id`

**Admissions**
- `admission_id`
- `patient_id`
- `hospital_id`

**Diagnoses**
- `diag_id`
- `admission_id`

**Billing**
- `bill_id`
- `admission_id`

**Hospitals**
- `hospital_id`


In [48]:
# Expected 8-4-4-4-12 identifier pattern

uuid_pattern = (
    r"^[a-zA-Z0-9]{8}-"
    r"[a-zA-Z0-9]{4}-"
    r"[a-zA-Z0-9]{4}-"
    r"[a-zA-Z0-9]{4}-"
    r"[a-zA-Z0-9]{12}$"
)

In [49]:
def check_id_format(
    df,
    column_name,
    dataset_name
):

    values = df[column_name]

    non_null = values.notna()

    invalid_ids = df[
        non_null
        &
        ~values.astype(str)
        .str.match(
            uuid_pattern,
            na=False
        )
    ]

    return {
        "Dataset": dataset_name,
        "Column": column_name,
        "Non-Null IDs": int(non_null.sum()),
        "Null IDs": int((~non_null).sum()),
        "Invalid Format": len(invalid_ids),
    }

In [50]:
id_fields = {

    "Patients": [
        "patient_id"
    ],

    "Admissions": [
        "admission_id",
        "patient_id",
        "hospital_id"
    ],

    "Diagnoses": [
        "diag_id",
        "admission_id"
    ],

    "Billing": [
        "bill_id",
        "admission_id"
    ],

    "Hospitals": [
        "hospital_id"
    ],
}

In [51]:
id_validation_results = []

for dataset_name, columns in id_fields.items():

    df = datasets[dataset_name]

    for column in columns:

        result = check_id_format(
            df,
            column,
            dataset_name
        )

        id_validation_results.append(result)


id_validation_summary = pd.DataFrame(
    id_validation_results
)

display(id_validation_summary)

,Dataset,Column,Non-Null IDs,Null IDs,Invalid Format
0,Patients,patient_id,86400,0,0
1,Admissions,admission_id,120000,0,0
2,Admissions,patient_id,120000,0,0
3,Admissions,hospital_id,120000,0,0
4,Diagnoses,diag_id,271341,0,0
5,Diagnoses,admission_id,271341,0,0
6,Billing,bill_id,120000,0,0
7,Billing,admission_id,120000,0,0
8,Hospitals,hospital_id,33,0,0


In [52]:
# Leading/ Trailing Space Check

def check_space_issue(
    df,
    column_name,
    dataset_name
):

    values = df[column_name]

    mask = (
        values.notna()
        &
        values.astype(str).ne(
            values.astype(str).str.strip()
        )
    )

    return {
        "Dataset": dataset_name,
        "Column": column_name,
        "Leading/Trailing Spaces": int(mask.sum()),
    }

In [53]:
space_results = []

for dataset_name, columns in id_fields.items():

    df = datasets[dataset_name]

    for column in columns:

        result = check_space_issue(
            df,
            column,
            dataset_name
        )

        space_results.append(result)


space_validation_summary = pd.DataFrame(
    space_results
)

display(space_validation_summary)

,Dataset,Column,Leading/Trailing Spaces
0,Patients,patient_id,0
1,Admissions,admission_id,0
2,Admissions,patient_id,0
3,Admissions,hospital_id,0
4,Diagnoses,diag_id,0
5,Diagnoses,admission_id,0
6,Billing,bill_id,0
7,Billing,admission_id,0
8,Hospitals,hospital_id,0


In [54]:
# 3.5. Hospital Uniqueness Check

# --------------------------------------------------
# Count unique hospital IDs and hospital names
# --------------------------------------------------

unique_hospital_id = (
    df_hospitals["hospital_id"]
    .nunique()
)

unique_hospital_name = (
    df_hospitals["name"]
    .nunique()
)

print(
    "Unique hospital_id:",
    unique_hospital_id
)

print(
    "Unique hospital name:",
    unique_hospital_name
)

Unique hospital_id: 33
Unique hospital name: 26


In [55]:
# --------------------------------------------------
# Check for duplicate hospital records
# excluding hospital_id
# --------------------------------------------------

hospital_attributes = [
    col
    for col in df_hospitals.columns
    if col != "hospital_id"
]

duplicate_hospitals = df_hospitals[
    df_hospitals.duplicated(
        subset=hospital_attributes,
        keep=False
    )
].sort_values(
    by=hospital_attributes
)

print(
    "Duplicate hospital records "
    "excluding hospital_id:",
    len(duplicate_hospitals)
)

if len(duplicate_hospitals) > 0:

    display(
        duplicate_hospitals
    )

Duplicate hospital records excluding hospital_id: 0


In [56]:
# --------------------------------------------------
# Check for hospital names appearing more than once
# --------------------------------------------------

duplicate_names = (
    df_hospitals["name"]
    .value_counts()
)

duplicate_names = (
    duplicate_names[
        duplicate_names > 1
    ]
)

print(
    "Hospital names appearing more than once:"
)

display(duplicate_names)

Hospital names appearing more than once:


,count
name,
Apollo/Manipal (Private) Andhra Pradesh,4
District Hospital Uttar Pradesh,2
District Hospital Odisha,2
District Hospital Bihar,2
Apollo/Manipal (Private) Karnataka,2


In [57]:
if len(duplicate_names) > 0:

    display(
        df_hospitals[
            df_hospitals["name"].isin(
                duplicate_names.index
            )
        ].sort_values(
            by="name"
        )
    )

,hospital_id,name,state,tier,beds,teaching
25,422f6ed3-147b-4f1a-815b-7a383a181a13,Apollo/Manipal (Private) Andhra Pradesh,Andhra Pradesh,tier1,663,True
27,fe7c70a4-5806-45e4-9ec8-3a329d41ee5a,Apollo/Manipal (Private) Andhra Pradesh,Andhra Pradesh,tier1,986,True
28,b08067df-ad81-4829-a46f-3332da7739e4,Apollo/Manipal (Private) Andhra Pradesh,Andhra Pradesh,tier1,573,True
31,c78a3d62-36c7-4f33-b76b-3a9020611d24,Apollo/Manipal (Private) Andhra Pradesh,Andhra Pradesh,tier1,963,True
29,eae7c9ec-65c8-4d48-92f3-7da7b3e2b227,Apollo/Manipal (Private) Karnataka,Karnataka,tier1,1018,True
30,ad0ba923-bc16-42a4-a49c-9f8a5ce6e8c1,Apollo/Manipal (Private) Karnataka,Karnataka,tier1,999,True
19,d7a59112-7a8d-4ebf-b0b0-b5908358a10a,District Hospital Bihar,Bihar,tier3,142,False
21,a86650ea-3517-4285-bc11-ffa2085a8df2,District Hospital Bihar,Bihar,tier3,99,False
15,13fff27b-ec37-4afc-b5b3-51fedbc8b57e,District Hospital Odisha,Odisha,tier3,146,False
24,72a121a0-e938-4373-8839-fc3a2a35fa6e,District Hospital Odisha,Odisha,tier3,108,False


### 4. Data Quality Summary



In [58]:
# Consolidated automated checks

summary_tables = {

    "Dataset Profile":
        profile_summary,

    "Date Validation":
        date_validation_summary,

    "Identifier Format":
        id_validation_summary,

    "Identifier Whitespace":
        space_validation_summary,
}


for title, table in summary_tables.items():

    print("=" * 80)
    print(title)
    print("=" * 80)

    display(table)

Dataset Profile


,Dataset,Rows,Columns,Missing Values,Missing (%),Duplicate Rows,Numeric Columns,Categorical Columns,Date Columns
0,Patients,86400,8,24715,3.58,0,3,4,0
1,Admissions,120000,17,0,0.00,0,9,8,0
2,Diagnoses,271341,6,0,0.00,0,1,5,0
3,Billing,120000,6,0,0.00,0,3,3,0
4,Hospitals,33,6,0,0.00,0,1,4,0


Date Validation


,Check,Count
0,Invalid admit_date,0
1,Invalid discharge_date,0
2,Discharge before admit,0


Identifier Format


,Dataset,Column,Non-Null IDs,Null IDs,Invalid Format
0,Patients,patient_id,86400,0,0
1,Admissions,admission_id,120000,0,0
2,Admissions,patient_id,120000,0,0
3,Admissions,hospital_id,120000,0,0
4,Diagnoses,diag_id,271341,0,0
5,Diagnoses,admission_id,271341,0,0
6,Billing,bill_id,120000,0,0
7,Billing,admission_id,120000,0,0
8,Hospitals,hospital_id,33,0,0


Identifier Whitespace


,Dataset,Column,Leading/Trailing Spaces
0,Patients,patient_id,0
1,Admissions,admission_id,0
2,Admissions,patient_id,0
3,Admissions,hospital_id,0
4,Diagnoses,diag_id,0
5,Diagnoses,admission_id,0
6,Billing,bill_id,0
7,Billing,admission_id,0
8,Hospitals,hospital_id,0


Note: Interpretation for Downstream Analysis

This profiling notebook is a data-readiness assessment, not the final clinical analysis.

Before moving into exploratory analysis, feature engineering, predictive modeling, and dashboard development, the following should be confirmed:

- Key identifiers are structurally valid.
- Required dates are parseable and logically ordered.
- Missingness is understood for important analytical fields.
- Duplicate records do not create unintended row multiplication.
- Categorical values are standardized or documented.
- Referential integrity between related tables has been validated.
- Variables used for predictive modeling are evaluated for potential leakage.
- The final modeling population and outcome definition are explicitly documented.

This notebook therefore serves as evidence of the data-quality and data-understanding phase of the consulting engagement.